In [164]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import osmnx as ox
import folium
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px
from plotly.offline import plot
from pyproj import Transformer


In [83]:
#import my data
df_2014_2016 = pd.read_csv('Dallas_Police_Public_Data_RMS_Incidents_with_GeoLocation_20260626.csv')
df_2018 = pd.read_csv('Police_Incidents_20260626.csv')

/tmp/ipykernel_3586388/1090212874.py:2: DtypeWarning:

Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.

/tmp/ipykernel_3586388/1090212874.py:3: DtypeWarning:

Columns (3,61,62,69,70,72) have mixed types. Specify dtype option on import or set low_memory=False.



In [129]:
# #explore to see what columns to keep
# df_2014_2016.info()

In [128]:
# df_2018.info()

In [154]:
#grab only a few columns
#grabbing only 2014-2016 values
df_select_2014 = df_2014_2016[df_2014_2016['Year1'] >= 2014.0]
df_select_2014 = df_select_2014[['IncidentNum', 'Year1', 'Address', 'AptNum','City', 'ZipCode', 'State', 'PointX', 'PointY']]

In [155]:
#grab only a few columns
#rename to match df_select_2014 so can concat later
df_select_2018 = df_2018[['Incident Number w/year', 'Year1 of Occurrence', 'Incident Address', 'Apartment Number', 'City', 'Zip Code', 'State', 'X Coordinate', 'Y Cordinate']]
df_select_2018 = df_select_2018.rename(columns = {'Incident Number w/year': 'IncidentNum', 'Year1 of Occurrence':'Year1', 'Incident Address': 'Address', 'Apartment Number': 'AptNum', 'Zip Code':'Zipcode', 'X Coordinate': 'PointX', 'Y Cordinate':'PointY'})
df_select_2018 = df_select_2018[(df_select_2018['Year1'] >= 2014)]

In [156]:
#checking what years are in the df
df_select_2014.value_counts('Year1')

Year1
2015.0    94382
2014.0    54189
2016.0    30912
Name: count, dtype: int64

In [157]:
# df_select_2018.head()

In [158]:
# df_select_2014.head()

In [159]:
all_crimes_df = pd.concat([df_select_2014, df_select_2018])
print('Length before de-dupe: ', len(all_crimes_df))
all_crimes_df = all_crimes_df.drop_duplicates(subset='IncidentNum')
print('Length after de-dupe: ', len(all_crimes_df))
all_crimes_df.head()
print("Count of all PointX NAs: ", len(all_crimes_df[all_crimes_df['PointX'].isna()]))
print("Count of all PointY NAs: ", len(all_crimes_df[all_crimes_df['PointX'].isna()]))

#okay seems like all rows missing PointX are also missing PointY 
#how would we like to handle the missingness? just drop? Get the points from the adresses? Shouldn't be too hard with OSMNX

Length before de-dupe:  1673759
Length after de-dupe:  1270410
Count of all PointX NAs:  7888
Count of all PointY NAs:  7888


In [160]:
all_crimes_df.value_counts('Year1')

Year1
2021.0    114485
2019.0    114264
2020.0    113552
2023.0    112989
2022.0    112926
2018.0    107334
2024.0    106876
2025.0    104451
2015.0     95759
2016.0     95319
2017.0     86813
2014.0     57047
2026.0     48595
Name: count, dtype: int64

In [165]:
#transforming from CRS:2276 to CRS:4326 (GPS for interactive map plotting) 
transformer = Transformer.from_crs(2276, 4326, always_xy=True)
all_crimes_df['lon'], all_crimes_df['lat'] = transformer.transform(all_crimes_df['PointX'], all_crimes_df['PointY'])

In [166]:
# df_select_2014.head()
all_crimes_df.head()

,IncidentNum,Year1,Address,AptNum,City,ZipCode,State,PointX,PointY,Zipcode,lon,lat
0,099903-2016,2016.0,7924 BRIARIDGE RD,NaN,DALLAS,75248.0,TX,2.498539e+06,7.035523e+06,NaN,-96.771801,32.957165
1,101365-2016,2016.0,1186 N ST AUGUSTINE DR,1075,DALLAS,75217.0,TX,2.535262e+06,6.954428e+06,NaN,-96.656727,32.732572
2,101239-2016,2016.0,12366 GARDEN GROVE DR,NaN,DALLAS,75253.0,TX,2.551475e+06,6.939565e+06,NaN,-96.604879,32.690932
3,101373-2016,2016.0,800 S ERVAY ST,NaN,DALLAS,75201.0,TX,2.492542e+06,6.969561e+06,NaN,-96.794840,32.776148
4,099558-2016,2016.0,700 S MADISON,NaN,DALLAS,75208.0,TX,2.483102e+06,6.955956e+06,NaN,-96.826258,32.739174


In [75]:
# #make the data frame into a geopandas df
# gdf = gpd.GeoDataFrame(
#     df_select_2014, 
#     geometry=gpd.points_from_xy(df_select_2014.PointX, df_select_2014.PointY), 
#     crs="EPSG:2276" # Replace with your exact existing CRS
# )

#I tried to convert the dataframe into a geopandas df, but realized I didn't need to for the plotting

In [171]:
#Just messing around with osmnx for plotting, used plotly instead below
# place_name = "Dallas, Texas, USA"

# api = ox.geocoder.geocode_to_gdf(place_name)

# m = api.explore(
#     style_kwds={"fillOpacity": 0.1}
# )

# gdf_gps = gdf.to_crs("EPSG:4326")

# # 3. Plot the transformed data
# gdf_gps.explore(
#     m=m, 
#     color="red"
# )
# # gdf.explore(
# #     m = m, 
# #     color="red" 
# #    # marker_kwds={"radius": 6, "fill": True},
# #    # popup=True 
# )

In [173]:
#making the figure object
fig = px.scatter_map(df_select_2014, lat="lat", lon="lon", color = 'Year1')

In [ ]:
# # DO NOT UNCOMMENT AND RUN THIS, WILL CRASH!! TOO MANY DATA POINTS TO ACTUALLY SHOW...
# fig